In [3]:
# ============================================
# 1. IMPORTS
# ============================================

from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

print("All imports successful.")

All imports successful.


In [4]:
# ============================================
# 2. PROJECT PATHS
# ============================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

ML_DATASET_PATH = PROCESSED_DIR / "ml_dataset.parquet"

print("Project root:", PROJECT_ROOT)
print("ML dataset:", ML_DATASET_PATH)
print("Models directory:", MODELS_DIR)

Project root: D:\01_PROJECTS\DS_AI\flood-prediction-india
ML dataset: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\processed\ml_dataset.parquet
Models directory: D:\01_PROJECTS\DS_AI\flood-prediction-india\models


In [5]:
# ============================================
# 3. LOAD ML DATASET
# ============================================

ml_dataset = pd.read_parquet(ML_DATASET_PATH)

ml_dataset["date"] = pd.to_datetime(ml_dataset["date"])

print("Dataset shape:", ml_dataset.shape)
print("Date range:", ml_dataset["date"].min(), "→", ml_dataset["date"].max())

Dataset shape: (3292158, 20)
Date range: 1964-06-19 00:00:00 → 2026-09-10 00:00:00


In [6]:
# ============================================
# 4. CREATE TEST SET
# ============================================

test_df = ml_dataset[
    ml_dataset["date"].dt.year >= 2025
].copy()

print("Test shape:", test_df.shape)
print("Test dates:", test_df["date"].min(), "→", test_df["date"].max())
print("\nTest target distribution:")
print(test_df["flood_next_day"].value_counts())

Test shape: (374322, 20)
Test dates: 2025-01-01 00:00:00 → 2026-09-10 00:00:00

Test target distribution:
flood_next_day
0.0    370043
1.0      4279
Name: count, dtype: int64


In [7]:
# ============================================
# 5. MODEL FEATURES
# ============================================

model_features = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "level_danger_ratio",
    "level_warning_ratio",
    "rise_danger_ratio_1d",
    "rise_danger_ratio_3d",
    "danger_level",
    "warning_level",
    "lat",
    "lon"
]

X_test = test_df[model_features].copy()
y_test = test_df["flood_next_day"].astype(int)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (374322, 17)
y_test: (374322,)


In [9]:
# ============================================
# 6. LOAD TRAINED MODELS
# ============================================

logistic_model = joblib.load(
    MODELS_DIR / "logistic_regression.joblib"
)

rf_model = joblib.load(
    MODELS_DIR / "random_forest.joblib"
)

xgb_model = joblib.load(
    MODELS_DIR / "xgboost.joblib"
)

print("All models loaded successfully.")

All models loaded successfully.


In [10]:
# ============================================
# 7. LOCKED VALIDATION THRESHOLDS
# ============================================

logistic_threshold = 0.739413
rf_threshold = 0.904376
xgb_threshold = 0.955972

print("Locked thresholds:")
print(f"Logistic Regression: {logistic_threshold}")
print(f"Random Forest:       {rf_threshold}")
print(f"XGBoost:             {xgb_threshold}")

Locked thresholds:
Logistic Regression: 0.739413
Random Forest:       0.904376
XGBoost:             0.955972


In [11]:
# ============================================
# 8. GENERATE TEST PROBABILITIES
# ============================================

# Logistic Regression
start_time = time.time()

logistic_test_prob = logistic_model.predict_proba(X_test)[:, 1]

logistic_test_time = time.time() - start_time


# Random Forest
start_time = time.time()

rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

rf_test_time = time.time() - start_time


# XGBoost
start_time = time.time()

xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]

xgb_test_time = time.time() - start_time


print("Test probability generation completed.\n")

print(f"Logistic Regression: {logistic_test_time:.2f} seconds")
print(f"Random Forest:       {rf_test_time:.2f} seconds")
print(f"XGBoost:             {xgb_test_time:.2f} seconds")

print("\nProbability shapes:")
print("Logistic Regression:", logistic_test_prob.shape)
print("Random Forest:      ", rf_test_prob.shape)
print("XGBoost:            ", xgb_test_prob.shape)

Test probability generation completed.

Logistic Regression: 0.11 seconds
Random Forest:       0.77 seconds
XGBoost:             0.50 seconds

Probability shapes:
Logistic Regression: (374322,)
Random Forest:       (374322,)
XGBoost:             (374322,)


In [12]:
# ============================================
# 9. APPLY LOCKED THRESHOLDS
# ============================================

logistic_test_pred = (
    logistic_test_prob >= logistic_threshold
).astype(int)

rf_test_pred = (
    rf_test_prob >= rf_threshold
).astype(int)

xgb_test_pred = (
    xgb_test_prob >= xgb_threshold
).astype(int)


print("Predicted flood counts:")
print(f"Logistic Regression: {logistic_test_pred.sum():,}")
print(f"Random Forest:       {rf_test_pred.sum():,}")
print(f"XGBoost:             {xgb_test_pred.sum():,}")

print("\nActual flood count:")
print(f"{y_test.sum():,}")

Predicted flood counts:
Logistic Regression: 7,001
Random Forest:       3,951
XGBoost:             4,176

Actual flood count:
4,279


In [13]:
# ============================================
# 10. FINAL TEST METRICS
# ============================================

def evaluate_model(name, y_true, y_prob, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    return {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


test_results = pd.DataFrame([
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_test_prob,
        logistic_test_pred
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        rf_test_prob,
        rf_test_pred
    ),
    evaluate_model(
        "XGBoost",
        y_test,
        xgb_test_prob,
        xgb_test_pred
    )
])

print(test_results.to_string(index=False))

              Model  ROC-AUC   PR-AUC  Precision   Recall       F1     TN   FP  FN   TP
Logistic Regression 0.990337 0.751386   0.532924 0.871933 0.661525 366773 3270 548 3731
      Random Forest 0.996244 0.932416   0.903569 0.834307 0.867558 369662  381 709 3570
            XGBoost 0.996458 0.932761   0.875239 0.854172 0.864577 369522  521 624 3655


In [14]:
# ============================================
# 11. FINAL MODEL COMPARISON TABLE
# ============================================

final_comparison = test_results[
    [
        "Model",
        "ROC-AUC",
        "PR-AUC",
        "Precision",
        "Recall",
        "F1"
    ]
].copy()

print(final_comparison.to_string(index=False))

              Model  ROC-AUC   PR-AUC  Precision   Recall       F1
Logistic Regression 0.990337 0.751386   0.532924 0.871933 0.661525
      Random Forest 0.996244 0.932416   0.903569 0.834307 0.867558
            XGBoost 0.996458 0.932761   0.875239 0.854172 0.864577


In [15]:
final_comparison.to_csv(
    TABLES_DIR / "final_model_comparison.csv",
    index=False
)

print(
    f"\nSaved to: "
    f"{TABLES_DIR / 'final_model_comparison.csv'}"
)


Saved to: D:\01_PROJECTS\DS_AI\flood-prediction-india\reports\tables\final_model_comparison.csv


In [16]:
# ============================================
# 12. CONFUSION MATRICES
# ============================================

confusion_matrices = {
    "Logistic Regression": confusion_matrix(
        y_test,
        logistic_test_pred
    ),
    "Random Forest": confusion_matrix(
        y_test,
        rf_test_pred
    ),
    "XGBoost": confusion_matrix(
        y_test,
        xgb_test_pred
    )
}

for model_name, matrix in confusion_matrices.items():
    print(f"\n{model_name}")
    print(matrix)


Logistic Regression
[[366773   3270]
 [   548   3731]]

Random Forest
[[369662    381]
 [   709   3570]]

XGBoost
[[369522    521]
 [   624   3655]]
